# Heliyon Revision — GPU Experiments

**First: Runtime → Change runtime type → T4 GPU.** Nothing here works well on CPU.

This notebook runs everything still outstanding for the revision. All other
experiments are already finished locally.

| § | Experiment | Answers | Time on a free T4 |
|---|---|---|---|
| 3 | **E5** — fairness: subsample curve + class-weight parity | R1.15, R1.16, R3.4, R6.5 | ~35 min |
| 4 | **E8b** — per-class SHAP, re-run on corrected labels | R3.3, R6.12 | 20–40 min |
| 5 | **E7** — FT-Transformer benchmark | R3.2 | 5–7 h |
| 6 | **E6** — CNN1D / BiLSTM hyperparameter search | R7.1 | 1–2 h |

Cells are ordered cheapest-first, so **Runtime → Run all** is the intended way
to use this. Total is ~8–11 h against a ~6 h daily GPU allowance, so expect it
to take two sittings.

**When the GPU quota runs out, that is fine.** Everything finished has already
been downloaded, and re-running Run all the next day skips completed work and
picks up where it stopped. The only thing lost is the cell that was mid-run.

---

## Setup — upload files directly into the session

Using the file browser on the left (📁 icon), create two folders and upload
into them from `Revision_R1\experiments\colab_upload\`:

```
/content/
├── code/    utils.py, e67_dl_colab.py,
│            e5_fairness_gpu.py, e8b_shap_multi.py     (4 files, 48 KB)
└── data/    cicids2017.parquet  (174 MiB)
             unsw_nb15.parquet   (105 MiB)
             ton_iot.parquet     (  1 MiB)
```

Folder names do not matter — cell 1 searches the session for all seven files
and reports where it found each. It also checks each dataset's exact byte count
and row count, so a truncated upload is caught immediately rather than
surfacing later as `Parquet magic bytes not found in footer`.

**If you are resuming**, also upload your most recent downloaded result CSVs
into `/content/results/` before running. Every script skips work already
recorded there, so finished folds are not repeated.

---

## Saving results — automatic

Results live in the session, and **every experiment cell downloads them when it
finishes**. No Drive mount, nothing to remember. The long E7 and E8b cells
download after each dataset or task, not just at the end.

Each zip contains everything finished so far, so only the most recent one
matters. The single gap: a cell interrupted mid-run cannot download, which is
why the long runs are split per dataset.

---

## What changed after the last session

Both fixes were measured, not assumed:

* **E5's MLP** took ~18 min per fit because a small network was fed 1024-row
  batches through a DataLoader, leaving the GPU idle. The fold now lives on the
  device and is batched by index slicing: **under a minute** per fit, same
  accuracy.
* **E7** took 86 min per fold on UNSW-NB15. Moving the fold onto the GPU plus
  mixed-precision training gave **2.2× at identical F1** (0.9754 → 0.9754). The
  epoch budget was then cut 50 → 20 after measuring that the extra epochs buy
  0.0014 F1 for 2.5× the compute.

Resume logic was also added to E6 and E7, which previously had none.

## 1. Check the GPU and locate your uploaded files

This searches the session for the six required files wherever you put them.

In [ ]:
import pathlib

!nvidia-smi --query-gpu=name,memory.total --format=csv

ROOT = pathlib.Path('/content')

# Exact sizes and row counts of the staged files, so a truncated or still-
# uploading file is caught here rather than as an opaque Arrow error later.
EXPECTED = {
    'cicids2017.parquet': (181_964_205, 2_000_000),
    'ton_iot.parquet':    (    972_970,   211_043),
    'unsw_nb15.parquet':  (110_339_448, 2_000_000),
}
NEED_CODE = ['utils.py', 'e67_dl_colab.py', 'e5_fairness_gpu.py',
             'e8b_shap_multi.py']

# 'drive' is skipped so the search stays fast even if Drive is mounted.
SKIP = {'sample_data', 'drive', '.config', '.ipynb_checkpoints', '__pycache__'}


def locate(fname):
    """Find a file anywhere in the session, ignoring Colab's own folders."""
    for p in ROOT.rglob(fname):
        if SKIP.isdisjoint(p.parts) and p.is_file():
            return p
    return None


found, problems = {}, []

for f in NEED_CODE:
    p = locate(f)
    if p:
        found[f] = p
        print(f'  [ok]      {f:24s} -> {p.parent}')
    else:
        problems.append(f'{f}: not found')
        print(f'  [MISSING] {f}')

for f, (exp_bytes, exp_rows) in EXPECTED.items():
    p = locate(f)
    if not p:
        problems.append(f'{f}: not found')
        print(f'  [MISSING] {f}')
        continue
    n = p.stat().st_size
    if n != exp_bytes:
        pct = 100 * n / exp_bytes
        problems.append(
            f'{f}: {n:,} bytes, expected {exp_bytes:,} ({pct:.1f}%)')
        state = 'incomplete upload' if n < exp_bytes else 'unexpected file'
        print(f'  [BAD SIZE] {f:24s} {n:,} / {exp_bytes:,} bytes '
              f'({pct:.1f}% - {state})')
        continue
    try:
        import pyarrow.parquet as pq
        rows = pq.read_metadata(p).num_rows
        if rows != exp_rows:
            problems.append(f'{f}: {rows:,} rows, expected {exp_rows:,}')
            print(f'  [BAD DATA] {f:24s} {rows:,} rows, expected {exp_rows:,}')
        else:
            found[f] = p
            print(f'  [ok]      {f:24s} -> {p.parent}  ({rows:,} rows)')
    except Exception as exc:
        problems.append(f'{f}: unreadable ({type(exc).__name__})')
        print(f'  [CORRUPT] {f:24s} {type(exc).__name__}: {exc}')

if problems:
    print('\n' + '=' * 70)
    print('PROBLEMS FOUND - fix these before running anything else:\n')
    for p in problems:
        print('  * ' + p)
    print('\nIf a size is short, the browser upload did not finish. Delete that')
    print('file in the file browser and upload it again, keeping the Colab tab')
    print('open and in the foreground until the progress ring completes.')
    print('\nIf a size is unexpected rather than short, you may be uploading the')
    print('older, larger copies. Re-copy them from')
    print('Revision_R1\\experiments\\colab_upload\\data\\ - they were repacked')
    print('(float32 + zstd) and are now 280 MiB in total rather than 425 MiB.')
    print('=' * 70)
    raise SystemExit('Upload incomplete or wrong files - see above.')

DATA_DIR = found['cicids2017.parquet'].parent
SRC_DIR = found['utils.py'].parent

print(f'\ndata : {DATA_DIR}')
print(f'code : {SRC_DIR}')
print('\nALL FILES PRESENT AND VALID  (results location is set in the next cell)')

### 1b. Results location and automatic checkpointing

Results are kept in the session at `/content/results` — no Drive mount.

`/content` is erased when the runtime disconnects, so the next cell defines a
`checkpoint()` helper that zips the results and downloads them to your
computer. **Every experiment cell calls it automatically when it finishes**, so
a completed dataset is on your machine before the next one starts. You can also
call `checkpoint()` by hand at any time.

What this does and does not protect:

* **Protected:** anything that finished. Each dataset downloads as soon as its
  cell completes.
* **Not protected:** a cell interrupted part-way. The long E7 cells run 2–3 h,
  and if the runtime dies at hour 2 the folds finished inside that cell are
  lost, because a running cell cannot download.

To narrow that window on the long runs, E7 is split so each cell covers one
dataset, and you can call `checkpoint()` yourself between them.

**When you resume in a later session:** upload the downloaded CSVs back into
`/content/results/` along with the data. Every script skips work already
recorded there, so it continues rather than restarting.

In [ ]:
# Results stay in the session; checkpoint() saves them out.
import shutil, datetime, time

RESULTS = ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
SAVES = ROOT / 'saved_zips'          # zips also kept here, in the file browser
SAVES.mkdir(parents=True, exist_ok=True)
print(f'results -> {RESULTS}')
print(f'zips    -> {SAVES}   (grab these from the file browser if a browser '
      f'download is blocked)')

_last_download = [0.0]
MIN_GAP_S = 90        # browsers block rapid-fire automatic downloads


def checkpoint(label='', force=False):
    """Zip the results and try to download them.

    The zip is ALWAYS written to /content/saved_zips, so it can be retrieved by
    hand from the file browser even when the browser refuses the automatic
    download. Chrome blocks multiple automatic downloads from one page, so
    calls are rate-limited; pass force=True to override.
    """
    items = sorted(p for p in RESULTS.glob('*') if p.is_file())
    if not items:
        print('checkpoint: nothing to save yet')
        return None

    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    name = f'ids_results_{label}_{stamp}' if label else f'ids_results_{stamp}'
    archive = shutil.make_archive(str(SAVES / name), 'zip', RESULTS)
    kb = pathlib.Path(archive).stat().st_size / 1024
    print(f'\ncheckpoint: {len(items)} file(s) -> {name}.zip ({kb:.0f} KB)')
    print(f'  saved in session at {archive}')

    gap = time.time() - _last_download[0]
    if not force and gap < MIN_GAP_S:
        print(f'  (browser download skipped - last one was {gap:.0f}s ago; '
              f'the next checkpoint will include this data anyway)')
        return archive

    try:
        from google.colab import files
        files.download(archive)
        _last_download[0] = time.time()
        print('  browser download started')
        print('  If nothing lands in Downloads, Chrome has blocked it: click')
        print('  the blocked-download icon in the address bar and Allow, or')
        print('  fetch the zip from the file browser (folder icon, left).')
    except Exception as exc:
        print(f'  browser download failed ({type(exc).__name__}) - '
              f'use the file browser to get {archive}')
    return archive


existing = sorted(p.name for p in RESULTS.glob('*.csv'))
print(f'\n{len(existing)} result file(s) already present'
      + (':' if existing else ' - starting fresh'))
for n in existing:
    print('   ', n)
if existing:
    print('\nCompleted work in these files is skipped, so re-running resumes.')
else:
    print('\nIf resuming a previous session, upload your most recent result')
    print(f'CSVs into {RESULTS} first, so finished work is not redone.')

## 2. Install cuML and load the code

cuML (RAPIDS) provides the GPU SVM / k-NN / Random Forest that E5 needs. It is
often already present on Colab; the cell below installs it only if it is not,
which takes a few minutes.

If the install fails, E5 still runs — it falls back to scikit-learn on CPU and
records `backend=sklearn` in every row — but Part B will be slow again, so
prefer to get cuML working.

Sections 4 and 5 (E6/E7) use PyTorch only and do not need cuML at all.

In [ ]:
import subprocess, sys, importlib

try:
    import cuml
    print('cuML already available:', cuml.__version__)
except ImportError:
    print('Installing cuML (a few minutes)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--extra-index-url=https://pypi.nvidia.com',
                    'cuml-cu12'], check=False)
    try:
        import cuml
        print('cuML installed:', cuml.__version__)
    except ImportError:
        print('cuML unavailable — E5 will fall back to scikit-learn (CPU, slow).')

In [ ]:
import sys, importlib

sys.path.insert(0, str(SRC_DIR))

import utils
importlib.reload(utils)
# utils.py derives its paths from __file__; point them at the session folders.
utils.DATA_CLEAN = DATA_DIR
utils.DATA_R1 = DATA_DIR          # dataset_path() checks here first
utils.TABLES = RESULTS
utils.FIGURES = RESULTS

def _rebind(mod, **names):
    """Modules bind DATA_CLEAN/TABLES at import time; rebind after reload."""
    importlib.reload(mod)
    mod.DATA_CLEAN = utils.DATA_CLEAN
    mod.TABLES = utils.TABLES
    if hasattr(mod, 'dataset_path'):
        mod.dataset_path = utils.dataset_path
    for k, v in names.items():
        setattr(mod, k, v)
    return mod

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('data  :', utils.DATA_CLEAN)
print('output:', utils.TABLES)

## 3. E5 — fairness of the model comparison

Two reviewer objections, two parts.

**Part B — the subsample curve** (R1.16, R3.4, R6.5): *"SVM and k-NN were
capped at 50k while other models saw 1.6M rows, which may understate them."*
Runs k-NN at 50k/100k/200k, RBF-SVM at 50k/100k under both the paper's
`max_iter=5000` and a raised 50,000 budget, and LinearSVC up to the full
training fold. Every fit records whether it hit its iteration cap, so a low
score can be attributed to non-convergence rather than to the data.

**Part A — imbalance-handling parity** (R1.15, R6.5): *"Why no class_weight
for RF/SVM, no weighted loss for MLP?"* Runs Random Forest and SVM with and
without balanced class weights, and an MLP with and without class-weighted
cross-entropy. The MLP is a PyTorch reimplementation of the benchmark's 128–64
network specifically so the weighted-loss arm can exist at all — scikit-learn's
`MLPClassifier` supports neither `class_weight` nor `sample_weight`, which is
why the submitted paper had no such arm.

Run Part B first (it is the slower and more contested of the two).

In [ ]:
# E5 Part B — subsample sensitivity curve
import e5_fairness_gpu as e5
e5 = _rebind(e5,
             OUT_A=utils.TABLES / 'e5_class_weight.csv',
             OUT_B=utils.TABLES / 'e5_subsample_curve.csv')
print('backend:', e5.BACKEND)   # 'cuml' = GPU, 'sklearn' = CPU fallback

e5.part_b()
checkpoint('e5b')

In [ ]:
# E5 Part A — imbalance parity. One dataset per cell: each is a bounded chunk,
# anything already recorded is skipped, and results download when the cell ends.
e5.part_a(datasets=['ton_iot'])          # fastest, ~3 min
e5.part_a(datasets=['unsw_nb15'])        # ~10 min
e5.part_a(datasets=['cicids2017'])       # slowest, ~15 min
checkpoint('e5a')

## 4. E8b — per-class SHAP attribution (re-run on corrected labels)

Answers R3.3 and R6.12: per-attack-class SHAP heatmaps, a correct-versus-
misclassified breakdown for the rare classes, and fold-to-fold ranking
stability.

This already ran locally, but **before** three label defects were fixed (the
duplicated UNSW-NB15 `Backdoor`/`Backdoors` classes, the mojibake CICIDS2017
web-attack names, and UNSW-NB15's benign class written as `None`). The numbers
are sound; only the class *labels* are stale, so it needs one clean re-run.

It sits here — before the long E7 — because it is short (20–40 min) and
completes a manuscript section outright. Fitting 15-class XGBoost on 2M rows is
slow on CPU and quick on a T4.

Installs `shap` if needed and downloads results after each dataset.

In [ ]:
# E8b — per-class SHAP. Checks its prerequisites first.
import subprocess, sys, importlib

if not (SRC_DIR / 'e8b_shap_multi.py').exists():
    raise SystemExit(
        f'Upload e8b_shap_multi.py into {SRC_DIR} first '
        '(it is in Revision_R1\\experiments\\colab_upload\\code\\).')

try:
    import shap
except ImportError:
    print('installing shap ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'],
                   check=False)
    import shap
print('shap', shap.__version__)

import e8b_shap_multi as e8b
e8b = _rebind(e8b, FIGURES=RESULTS)
print('xgboost device kwargs:', e8b._xgb_device())

stabs, mis = [], []
import pandas as pd
for ds in ['ton_iot', 'unsw_nb15', 'cicids2017']:   # cheapest first
    print(f'=== E8b {ds} ===')
    s, m = e8b.run_dataset(ds)
    stabs.append(s)
    mis.extend(m)
    pd.DataFrame(stabs).to_csv(RESULTS / 'e8b_shap_stability.csv', index=False)
    pd.DataFrame(mis).to_csv(RESULTS / 'e8b_shap_misclassified.csv', index=False)

# One download for the whole section: each zip is cumulative anyway, and
# firing several in quick succession is what browsers block.
checkpoint('e5_e8b', force=True)

## 5. E7 — FT-Transformer benchmark

Answers R3.2's request for a state-of-the-art tabular deep-learning baseline.
Unlike the 1D-CNN and BiLSTM, the FT-Transformer is permutation-equivariant
over features, so it does not depend on the arbitrary column ordering that
Reviewer 1 questioned.

**This is the long pole — expect it to span two sessions.** Each cell is one
dataset, every fold is saved as it completes, and results download after each
task. Re-running a finished cell prints "already complete" and costs nothing.

| Dataset | Both tasks, 5 folds |
|---|---|
| ToN-IoT | 15–25 min |
| UNSW-NB15 | 2–3 h |
| CICIDS2017 | 2.5–4 h |

> **Why this got faster.** The first attempt took 86 minutes *per fold* on
> UNSW-NB15. Two changes fixed that without touching the model: the fold now
> lives on the GPU and is batched by index slicing rather than streamed from
> host memory through a DataLoader, and training runs in mixed precision
> (evaluation stays fp32). Together, 2.2× at identical F1 (0.9754 → 0.9754).
>
> The epoch budget was then cut from 50 to 20 after measuring what it buys:
> 0.0014 macro F1 for 2.5× the compute, with validation loss flat well before
> epoch 20. That was preferred over subsampling the data or dropping folds,
> which would have saved similar time but weakened the comparison itself.

In [ ]:
import e67_dl_colab as dl
dl = _rebind(dl,
             OUT_E6=utils.TABLES / 'e6_dl_tuning.csv',
             OUT_E6_FOLDS=utils.TABLES / 'e6_dl_tuned_folds.csv',
             OUT_E7=utils.TABLES / 'e7_ft_transformer.csv')
print('device:', dl.DEVICE, '| max_epochs:', dl.MAX_EPOCHS,
      '| patience:', dl.PATIENCE)

# ToN-IoT — the quick one (15-25 min for both tasks)
for task in ['binary', 'multi']:
    dl.run_e7('ton_iot', task)
checkpoint('e7_toniot')

In [ ]:
# UNSW-NB15 — 2-3 h for both tasks. Safe to interrupt; re-run to continue.
# Downloads after each task so a disconnect during multi-class cannot cost
# you the binary results.
for task in ['binary', 'multi']:
    dl.run_e7('unsw_nb15', task)
    checkpoint(f'e7_unsw_{task}')

In [ ]:
# CICIDS2017 — 2.5-4 h for both tasks. Give this its own session.
# Downloads after each task rather than only at the end, so a disconnect
# during the multi-class run cannot cost you the binary results.
for task in ['binary', 'multi']:
    dl.run_e7('cicids2017', task)
    checkpoint(f'e7_cicids_{task}')

## 6. E6 — CNN1D / BiLSTM hyperparameter search

Answers R7.1: the submitted paper used fixed hyperparameters for both deep
models, which makes the tree-versus-deep comparison unfair given how much more
hyperparameter-sensitive deep models are.

20 random configurations (learning rate, width/depth, dropout, weight decay,
batch size) on a held-out split capped at 400k rows, then the best
configuration is re-evaluated with the full 5-fold protocol so it is directly
comparable to the submitted paper's numbers.

The multi-class task is where the objection bites hardest, so those
combinations come first. Each cell downloads when it finishes, and the search
resumes trial-by-trial if interrupted.

In [ ]:
# Loader for E6. Self-contained, so section 5 (E7) can be skipped entirely -
# useful when E7 is being run elsewhere.
import e67_dl_colab as dl
dl = _rebind(dl,
             OUT_E6=utils.TABLES / 'e6_dl_tuning.csv',
             OUT_E6_FOLDS=utils.TABLES / 'e6_dl_tuned_folds.csv',
             OUT_E7=utils.TABLES / 'e7_ft_transformer.csv')
print('device:', dl.DEVICE, '| max_epochs:', dl.MAX_EPOCHS)

dl.run_e6('ton_iot', 'multi', 'CNN1D')
checkpoint('e6_toniot_cnn')

In [ ]:
dl.run_e6('ton_iot', 'multi', 'BiLSTM')
checkpoint('e6_toniot_lstm')

In [ ]:
dl.run_e6('unsw_nb15', 'multi', 'CNN1D')
checkpoint('e6_unsw_cnn')

In [ ]:
dl.run_e6('unsw_nb15', 'multi', 'BiLSTM')
checkpoint('e6_unsw_lstm')

## 7. Check progress / save on demand

The first cell shows what has finished. The second saves on demand — the
experiment cells already download automatically, so this is just for peace of
mind before closing the tab.

Unzip downloads into `Revision_R1\results_r1\` — CSVs into `tables\`, PDFs into
`figures\`. Each zip contains everything finished up to that point, so you only
need to keep the most recent one.

| File | Feeds manuscript section |
|---|---|
| `e5_subsample_curve.csv` | Fairness — subsample curve |
| `e5_class_weight.csv` | Fairness — imbalance parity |
| `e8b_shap_perclass_*.csv`, `e8b_shap_stability.csv`, `e8b_shap_misclassified.csv` | Per-class SHAP |
| `fig_shap_multiclass_*.pdf` | Per-class SHAP heatmaps |
| `e7_ft_transformer.csv` | Deep-model comparison |
| `e6_dl_tuning.csv`, `e6_dl_tuned_folds.csv` | Deep-model tuning |

In [ ]:
import pandas as pd

CHECKS = [
    ('e5_subsample_curve.csv', ['dataset', 'model', 'config', 'n_train']),
    ('e5_class_weight.csv',    ['dataset', 'task', 'model', 'config']),
    ('e7_ft_transformer.csv',  ['dataset', 'task', 'model']),
    ('e6_dl_tuned_folds.csv',  ['dataset', 'task', 'model']),
    ('e6_dl_tuning.csv',       ['dataset', 'task', 'model']),
]

for name, group in CHECKS:
    p = utils.TABLES / name
    if not p.exists():
        print(f'\n--- {name}: not started ---')
        continue
    df = pd.read_csv(p)
    group = [g for g in group if g in df.columns]
    print(f'\n--- {name}  ({len(df)} rows) ---')
    agg = df.groupby(group)['f1_macro'].agg(['count', 'mean']).round(4)
    if 'converged' in df.columns:
        agg['converged'] = df.groupby(group)['converged'].agg(
            lambda s: s.dropna().all() if s.notna().any() else None)
    print(agg.to_string())

In [ ]:
# === MANUAL SAVE — run this any time, e.g. before closing the tab ===
# force=True bypasses the rate limit, so this always produces a fresh zip.
path = checkpoint('manual', force=True)

# Every zip written this session, newest last. If a browser download was
# blocked, download any of these by hand: file browser (folder icon, left)
# -> content -> saved_zips -> right-click -> Download.
print('\nzips available in the session:')
for z in sorted(SAVES.glob('*.zip')):
    print(f'  {z.name:52s} {z.stat().st_size/1024:7.0f} KB')
print('\nOnly the newest matters - each contains everything finished so far.')